# DSLM — train, save, and query

Selected model: `HuggingFaceTB/SmolLM2-135M-Instruct`.

The loop is the usual domain-SLM notebook loop:

1. Write domain rows as chat JSONL.
2. Render them with the model chat template.
3. LoRA-finetune, with loss only on assistant tokens.
4. Save the adapter and a manifest.
5. Query with the same template.

Run these cells from the repository root, with the package installed (`pip install -e .`).

## Chat template sample

SmolLM2 is ChatML. A training row becomes:

```
<|im_start|>system
You are Harborline, the policy assistant for Harborline Logistics. Answer only from Harborline policy. If a rule is not in the policy, say so.<|im_end|>
<|im_start|>user
What is the cutoff for same-day dispatch from the Pune hub?<|im_end|>
<|im_start|>assistant
Same-day dispatch from the Pune hub closes at 14:30 IST. Orders released after 14:30 IST leave on the next business day.<|im_end|>
```

The JSONL object is `{"messages": [{"role": "system"|"user"|"assistant", "content": "..."}]}`. `instruction` / `output` rows are converted to that shape.

In [ ]:
from dslm.config import load_config
from dslm.template import sample_training_text
from dslm.train import validate

cfg = load_config("configs/smollm2-135m-instruct.yaml")
sample = sample_training_text(cfg["model"]["name"])
print(sample["model_name"], sample["source"])
print(sample["rendered"])
print("--- generation prefix ---")
print(sample["generation_prefix"])
validate(cfg)

## Train and save

The default config trains for 25 epochs, which is enough for the 12-row sample to be memorized on CPU (about a minute). A finished adapter is already at `runs/smollm2-harborline`. For a larger corpus, lower the learning rate toward `2e-4`.

`configs/smollm2-135m-instruct.smoke.yaml` is two steps only, to check that saving works. The adapter is written to `train.output_dir / adapter`, with `manifest.json` beside it.

In [ ]:
from dslm.train import train

# Uncomment the next two lines to train. The smoke config finishes in a few optimizer steps.
# smoke = load_config("configs/smollm2-135m-instruct.smoke.yaml")
# print(train(smoke))

## Query

Point `run_dir` at a directory that contains `manifest.json`. Leave it as `None` to query the base model.

In [ ]:
from dslm.query import load_bundle

run_dir = None  # e.g. "runs/smollm2-harborline"
# bundle = load_bundle(cfg, run_dir)
# messages = bundle.prepare_messages(question="What is the cutoff for same-day dispatch from the Pune hub?")
# result = bundle.generate(messages)
# print(result["prompt"])
# print("---")
# print(result["answer"])